# RiskHalo - Behavioral Risk Intelligence Engine

This notebook walks through each step of the RiskHalo pipeline. Run cells in order to parse trade data, compute behavioral metrics, build session summaries, and optionally query the coach agent.

**Prerequisites:**
- `OPENAI_API_KEY` in `.env` (required for embeddings and coach agent)
- `TAVILY_API_KEY` in `.env` (required for coach agent external search)

## Setup & Imports

Configure Python path and working directory so imports and file paths work correctly.

In [1]:
import glob
import os

DECLARED_RISK = 2000
DATA_FOLDER = "data/Weekly_Trade_Data_Uploads"
# --------------------------------
# Find all weekly Excel files (exclude Excel lock files ~$*.xlsx)
# --------------------------------
all_xlsx = glob.glob(os.path.join(DATA_FOLDER, "*.xlsx"))
file_paths = [
    f for f in all_xlsx
    if not os.path.basename(f).startswith("~$")
]

if not file_paths:
    print("No weekly trade files found.")

print(f"\n Found {len(file_paths)} weekly trade files.")


 Found 22 weekly trade files.


## Step 1: Parse Trade Data

Load and parse the broker export (Excel/CSV) into RiskHalo standardized schema.

In [8]:
from app.parser import TradeParser

parser = TradeParser(FILE_PATH)
df = parser.parse()

print(f"Parsed {len(df)} trades.All entry, exit prices, profits and losses are denominated in Indian Rupees (INR).")
df.head(10)

ValueError: Error reading file: [Errno 2] No such file or directory: 'data/Weekly_Trade_Data_Uploads/Week_1_Trade_data.xlsx'

## Step 2: Feature Engine

The Feature Engine turns trading results into measurable emotional-state indicators. Compute R-multiples, win/loss flags, streaks, and post-loss indicators.

In [3]:
import pandas as pd
from app.feature_engine import FeatureEngine

pd.set_option("display.expand_frame_repr", False)

feature_df = FeatureEngine(df, DECLARED_RISK).run()

print("""Feature descriptions:
 - R_proxy — The profit or loss of a trade expressed as a multiple of your declared risk per trade (e.g., +2R means you made twice what you risked).
 - is_win — Indicates whether the trade closed in profit (1 if profitable, 0 otherwise).
 - is_loss — Indicates whether the trade closed in loss (1 if losing trade, 0 otherwise).
 - is_breakeven — Indicates whether the trade closed near zero profit or loss (1 if breakeven, 0 otherwise).
 - loss_streak — The number of consecutive losing trades leading up to and including the current trade.
 - win_streak — The number of consecutive winning trades leading up to and including the current trade.
 - post_loss_flag — Marks whether the trade occurred immediately after a losing trade, used to detect emotional behavior shifts.
""")

print("Features computed:")
print(feature_df[["trade_sequence", "R_proxy", "is_win", "is_loss", "is_breakeven", "loss_streak", "win_streak", "post_loss_flag"]].head(15))

Feature descriptions:
 - R_proxy — The profit or loss of a trade expressed as a multiple of your declared risk per trade (e.g., +2R means you made twice what you risked).
 - is_win — Indicates whether the trade closed in profit (1 if profitable, 0 otherwise).
 - is_loss — Indicates whether the trade closed in loss (1 if losing trade, 0 otherwise).
 - is_breakeven — Indicates whether the trade closed near zero profit or loss (1 if breakeven, 0 otherwise).
 - loss_streak — The number of consecutive losing trades leading up to and including the current trade.
 - win_streak — The number of consecutive winning trades leading up to and including the current trade.
 - post_loss_flag — Marks whether the trade occurred immediately after a losing trade, used to detect emotional behavior shifts.

Features computed:
    trade_sequence    R_proxy  is_win  is_loss  is_breakeven  loss_streak  win_streak  post_loss_flag
0                1  -0.720045       0        1             0            1         

## Step 3: Behavioral Diagnosis

This classification is fully deterministic. The system compares performance in neutral vs post-loss states and computes distortion percentages before assigning a behavioral state. Classify behavioral state (STABLE, LOSS_ESCALATION, CONFIDENCE_CONTRACTION, ADAPTIVE_RECOVERY).

In [4]:
from app.behavioral_engine import BehavioralEngine

diagnosis = BehavioralEngine(feature_df).run()

print("""Diagnosis term descriptions:
- behavioral_state — This classifies your dominant behavioral pattern.
   1. STABLE — Your performance stays consistent regardless of recent losses, indicating disciplined and emotionally steady execution.
   2. LOSS_ESCALATION — Your losses become larger after losing trades, suggesting emotional risk escalation or revenge trading.
   3. CONFIDENCE_CONTRACTION — Your winning trades shrink after losses, indicating hesitation or reduced conviction.
   4. ADAPTIVE_RECOVERY — Your performance improves after losses, showing resilience and controlled execution under pressure.
- severity_score — This measures how strongly your performance changes after losses — closer to 0 means stable, closer to 1 means highly distorted.
- avg_R_normal — This is your average risk-adjusted return per trade under normal conditions.
- avg_R_post — This is your average risk-adjusted return on trades taken immediately after a loss.
- avg_win_R_normal — This shows the average size of your winning trades during stable conditions.
- avg_win_R_post — This shows the average size of your winning trades after a loss — useful to detect hesitation.
- avg_loss_R_normal — This shows the average size of your losing trades during normal conditions.
- avg_loss_R_post — This shows the average size of your losing trades after a loss — useful to detect escalation.
- win_rate_normal — This is your win percentage during normal trading conditions.
- win_rate_post — This is your win percentage immediately after a loss.
- R_drop_percent — This measures how much your overall trade performance shifts after a loss.
- win_shrink_percent — This shows whether your wins become smaller after losing trades.
- loss_expansion_percent — This shows whether your losses become larger after losing trades.
""")
print("Behavioral Diagnosis:")
for k, v in diagnosis.items():
    print(f"  {k}: {v}")

Diagnosis term descriptions:
- behavioral_state — This classifies your dominant behavioral pattern.
   1. STABLE — Your performance stays consistent regardless of recent losses, indicating disciplined and emotionally steady execution.
   2. LOSS_ESCALATION — Your losses become larger after losing trades, suggesting emotional risk escalation or revenge trading.
   3. CONFIDENCE_CONTRACTION — Your winning trades shrink after losses, indicating hesitation or reduced conviction.
   4. ADAPTIVE_RECOVERY — Your performance improves after losses, showing resilience and controlled execution under pressure.
- severity_score — This measures how strongly your performance changes after losses — closer to 0 means stable, closer to 1 means highly distorted.
- avg_R_normal — This is your average risk-adjusted return per trade under normal conditions.
- avg_R_post — This is your average risk-adjusted return on trades taken immediately after a loss.
- avg_win_R_normal — This shows the average size of y

## Step 4: Expectancy & Financial Impact

Compute expectancy shifts and estimate rupee impact of behavioral distortion.

In [5]:
from app.expectancy_engine import ExpectancyEngine, format_expectancy_summary

post_loss_trade_count = feature_df["post_loss_flag"].sum()
impact = ExpectancyEngine(diagnosis, DECLARED_RISK, len(feature_df), post_loss_trade_count).run()

summary = format_expectancy_summary(
    expectancy_normal_R=impact["expectancy_normal_R"],
    expectancy_post_R=impact["expectancy_post_R"],
    expectancy_delta_R=impact["expectancy_delta_R"],
    economic_impact_rupees=impact["economic_impact_rupees"],
    risk_per_trade=impact.get("risk_per_trade"),
)
print(summary)

print(f"""
Metric descriptions:
- expectancy_normal_R ({impact['expectancy_normal_R']})
   This is your average return per trade in risk units during normal trading conditions.

- expectancy_post_R ({impact['expectancy_post_R']})
   This is your average return per trade in risk units immediately after a losing trade.

- expectancy_delta_R ({impact['expectancy_delta_R']})
   This measures how much your performance changes after a loss — positive means improvement, negative means deterioration.

- economic_impact_rupees ({impact['economic_impact_rupees']})
   This estimates how much money your post-loss behavior added or cost you over the analyzed period.
""")

Performance Impact

Normal trades: -1.23R (~₹-2460 per trade)
After losses: 0.07R (~₹140 per trade)
Behavioral shift: 1.3R per trade
Estimated impact over period: ₹15648

Metric descriptions:
- expectancy_normal_R (-1.23)
   This is your average return per trade in risk units during normal trading conditions.

- expectancy_post_R (0.07)
   This is your average return per trade in risk units immediately after a losing trade.

- expectancy_delta_R (1.3)
   This measures how much your performance changes after a loss — positive means improvement, negative means deterioration.

- economic_impact_rupees (15648.0)
   This estimates how much money your post-loss behavior added or cost you over the analyzed period.



## Step 5: Build Session Summary

Create structured snapshot and narrative summary ready for embedding.

In [ ]:
from app.session_summary_builder import SessionSummaryBuilder
from app.rule_engine import RuleComplianceEngine
from app.config_loader import load_rules_config

# Rule compliance layer
rules_config = load_rules_config()
rule_engine = RuleComplianceEngine(feature_df, rules_config)
rule_results = rule_engine.run()

# Build full session snapshot (behavior + expectancy + rules)
builder = SessionSummaryBuilder(feature_df, diagnosis, impact, rule_results)
snapshot = builder.run()

print("Session ID:", snapshot["structured_summary"]["session_id"])
print("\nNarrative Summary:")
print(snapshot["narrative_summary"])
print("\nRule Narrative:")
print(snapshot["rule_narrative"])

Session ID: session_cb8bd5333a04

Narrative Summary:
In this session of 14 trades, you were classified as ADAPTIVE_RECOVERY (severity 0.09).

You tend to improve execution after losses rather than deteriorate. This reflects emotional resilience and controlled risk behavior.

Performance Impact

Normal trades: -1.23R (~₹-2460 per trade)
After losses: 0.07R (~₹140 per trade)
Behavioral shift: 1.3R per trade
Estimated impact over period: ₹15648


## Step 6: Process All Files & Store in ChromaDB

Processes **each** weekly trade file in `DATA_FOLDER` (same as `main.py`): parse → features → behavioral diagnosis → expectancy → rule compliance → session summary → embed → store in ChromaDB with metadata. **Requires OPENAI_API_KEY.**

In [3]:
#Ballu
import os
import sys
import traceback

# Ensure project root (where `app/` lives) is on sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "RiskHalo_Certification_Challenge"))
if os.path.isdir(PROJECT_ROOT) and PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from app.parser import TradeParser
from app.feature_engine import FeatureEngine
from app.behavioral_engine import BehavioralEngine
from app.expectancy_engine import ExpectancyEngine
from app.session_summary_builder import SessionSummaryBuilder
from app.rule_engine import RuleComplianceEngine
from app.config_loader import load_rules_config
from rag.embedder import OpenAIEmbedder
from rag.vector_store import RiskHaloVectorStore

embedder = OpenAIEmbedder()
vector_store = RiskHaloVectorStore()
rules_config = load_rules_config()
print(f"Found {len(FILE_PATHS)} weekly trade file(s). Processing each and storing in ChromaDB.\n")

for file_path in FILE_PATHS:
    try:
        print(f"Processing file: {file_path}")
        parser = TradeParser(file_path)
        df = parser.parse()
        feature_df = FeatureEngine(df, DECLARED_RISK).run()
        diagnosis = BehavioralEngine(feature_df).run()
        post_loss_trade_count = int(feature_df["post_loss_flag"].sum())
        impact = ExpectancyEngine(
            diagnosis, DECLARED_RISK, len(feature_df), post_loss_trade_count
        ).run()
        rule_engine = RuleComplianceEngine(feature_df, rules_config)
        rule_results = rule_engine.run()
        builder = SessionSummaryBuilder(feature_df, diagnosis, impact, rule_results)
        snapshot = builder.run()
        session_id = snapshot["structured_summary"]["session_id"]
        narrative = snapshot["narrative_summary"]
        rule_narrative = snapshot["rule_narrative"]
        full_narrative = narrative + "\n\n" + rule_narrative
        embedding = embedder.embed_text(full_narrative)
        metadata = {
            "behavioral_state": diagnosis["behavioral_state"],
            "severity_score": diagnosis["severity_score"],
            "discipline_score": rule_results["discipline_scores"]["overall_discipline_score"],
            "risk_breach_count": rule_results["violations"]["risk_breach_count"],
            "rr_violation_count": rule_results["violations"]["rr_violation_count"],
            "overtrading_days": rule_results["violations"]["overtrading_days"],
            "daily_loss_breaches": rule_results["violations"]["daily_loss_breaches"],
        }
        vector_store.add_session(
            session_id=session_id,
            embedding=embedding,
            document=full_narrative,
            metadata=metadata,
        )
        print(f"  Stored session_id: {session_id}")
    except Exception as e:
        print(f"  Error processing {file_path}: {e}")
        traceback.print_exc()
print("\nAll weekly files processed.")

ModuleNotFoundError: No module named 'app'

In [ ]:
#Bharath

## Step 7: Coach Agent (Optional)

Query the coach agent for behavioral insights. **Requires OPENAI_API_KEY and TAVILY_API_KEY.**

In [ ]:
from agents.coach_agent import ask_coach

# Example queries (uncomment to run)
# response = ask_coach("Am I improving over time?")
# print(response)

# response = ask_coach("Why do traders escalate after losses?")
# print(response)

response = ask_coach("Summarize my behavioral state from the latest session.")
print(response)

## Step 8: RAGAS Evaluation

Runs RAGAS metrics (faithfulness, context recall, response relevancy, etc.) using **persona-driven testsets** from Chroma session summaries. Requires sessions in Chroma (run Step 6 first). Same logic as `evaluation/run_ragas_evaluation.py`.

In [5]:
# Bharath
# Load all narrative summaries from ChromaDB and build `raw_docs`

import chromadb

# Connect to the same ChromaDB path used elsewhere (reuse existing settings)
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="riskhalo_sessions")

# Fetch all stored documents (session summaries)
data = collection.get(include=["documents"])
documents = data.get("documents", [])

# Chroma sometimes returns a list of lists; flatten if needed
if documents and isinstance(documents[0], list):
    documents = documents[0]

# Join all summaries into a single string, separated by two newlines
raw_docs = "\n\n".join(documents)

print(f"Loaded {len(documents)} summaries into raw_docs")

Loaded 0 summaries into raw_docs


In [3]:
from evaluation.ragas_eval import RiskHaloRagasEvaluator
from evaluation.personas import TRADER_PERSONAS

evaluator = RiskHaloRagasEvaluator()

# Evaluate one persona (e.g. revenge_trader); use list(TRADER_PERSONAS.keys()) for all
persona_name = "revenge_trader"
print(f"Evaluating persona: {persona_name}")
results = evaluator.evaluate_persona(persona_name)

print("\nRAGAS metrics:")
for metric, value in results.items():
    print(f"  {metric}: {round(value, 4)}")

Evaluating persona: revenge_trader
generate_persona_dataset::Loaded 28 summaries and 28 metadatas for persona: revenge_trader
generate_persona_dataset::Generated 28 rows for persona: revenge_trader
prepare_ragas_dataset::rag_result: {'question': 'Why do my losses increase after a losing trade?', 'retrieved_contexts': ['\nIn this session of 22 trades, you were classified as LOSS_ESCALATION (severity 1).\n\nLosses expand following losing trades, indicating emotional risk escalation. Reducing downside variance after drawdowns should be a priority.\n\nPerformance Impact\n\nNormal trades: 0.04R (~₹80 per trade)\nAfter losses: -0.1R (~₹-200 per trade)\nBehavioral shift: -0.13R per trade\nEstimated impact over period: ₹-2634\n\n\nExecution discipline was inconsistent this week (Score: 56.29/100), with structural rule breaches affecting stability. 5 trade(s) exceeded declared risk, indicating lapses in loss containment. 1 day(s) breached maximum daily loss, increasing capital volatility. Overt

KeyboardInterrupt: 

---
**End of pipeline.** Run all cells in order to reproduce the full flow.